In [ ]:
import csv

def gen_seats(first,ratings):
    n = int(len(ratings.keys())**0.5)
    seats = [None]*(n*n)
    seats[0] = first

    def recurse(index,n,seats,cache=[]):

        if index == n*n:
            cache.append(seats[:])

            return cache

        for person in sorted(list(ratings.keys())):
            if person not in seats:
                seats[index] = person
                cache = recurse(index+1,n,seats,ratings,cache)
                seats[index] = None

        return cache

    possible = recurse(1,n,seats,ratings)


    with open("possibleSeats.csv","w",newline="\n") as f:
        writer = csv.writer(f)
        writer.writerows(sorted(possible))


ratings = read_file()
gen_seats("A",ratings)

In [20]:
import csv

def read_file():
    with open("ratings.csv") as f:
        data = list(csv.reader(f))[1:]

    names = "ABCDEFGHI"

    ratings = {i:{} for i in names}

    for i in data:
            ratings[i[0]][i[1]] = int(i[-1])

    return ratings

             
def cal_hap_2(seats,n,ratings):
    tot_hap = 0

    grid = [seats[i:i+n] for i in range(0,n*n,n)]

    transposed = list(zip(*grid))

    combi = [
         grid,
         transposed,
         [list(reversed(i)) for i in grid],
         [list(reversed(i)) for i in transposed]
    ]

    for grid_var in combi:

        for i in range(n):

            for j in range(n-1):

                pov = grid_var[i][j]
                ref = grid_var[i][j+1]

                if pov is not None:
                    pov_ratings = ratings[pov]
                    tot_hap += pov_ratings.get(ref,0)

    return tot_hap


def cal_hap_1(seats, n, ratings):
    tot_hap = 0

    def calculate(i, delta):
        new_i = i + delta

        if not (0 <= new_i < len(seats)):
            return 0

        if delta in [-1, 1] and i // n != new_i // n:
            return 0

        their = seats[i]
        them = seats[new_i]

        if their is not None:
            return ratings[their].get(them, 0)

        return 0

    for student in range(len(seats)):
        for delta in [-n, -1, 1, n]:
            tot_hap += calculate(student, delta)

    return tot_hap
     

def happiest_seating(first,ratings):

    n = int(len(ratings.keys())**0.5)

    seats = [None]*(n*n)

    seats[0] = first

    def _helper(index,seats,n,ratings,best_hap=0,best_seats = None):

        if index == n*n:
            happiness = cal_hap_1(seats,n,ratings)

            if happiness > best_hap:
                best_hap = happiness
                best_seats = seats[:]

                return best_hap,best_seats

        for person in sorted(list(ratings.keys())):
            if person not in seats:
                seats[index] = person
                best_hap,best_seats = _helper(index+1,seats,n,ratings,best_hap,best_seats)
                seats[index] = None


        return best_hap,best_seats


    
    best_hap,best_seats = _helper(1,seats,n,ratings)

    grid = [best_seats[i:i+n] for i in range(0,n*n,n)]

    return grid, best_hap



ratings = read_file()
grid,score = happiest_seating("A",ratings)

print("GRID")
print(*grid,sep='\n')
print()
print("SCORE")
print(score)

GRID
['A', 'B', 'C']
['H', 'I', 'D']
['F', 'G', 'E']

SCORE
90
